# Анализ аудитории витуберов

Посмотрим ключевую статистику комьюнити витуберов:

**Что сделано**:

* Рост аудитории
* Уникальность витуберов
* Кластеризация витуберов
* Количество локальных фолловов у зрителей выборки
* Количество глобальных фолловов у зрителей выборки
* Граф самых близости витуберов (то есть если косинусное расстояние меньше какого-то порога, то рисовать ребро, иначе - нет)

**Что пока нет**:

* Размер активной аудитории
* Активность чата
* Кластеризация сообщений в чате

#### Импорт всего нужного

In [1]:
import os
import csv
import sys
sys.path.insert(1, '../util/')

from data import UserData, FollowerData, json_to_user_data
from userdata import get_userdata, get_userdata_by_login
from followers import get_followers

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import numpy as np
import sklearn
from datetime import datetime
from typing import Optional, List, Tuple, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

## Prepare data

In [2]:
VTUBERS_LIST_FILE_PATH = "./data/vtubers.txt"

In [3]:
@dataclass
class VtubersData:
    vtuber_names: List[str] # 1 x N
    vtuber_followers: Dict[str, List[FollowerData]]
    user_following: Dict[int, List[str]]

In [4]:
vtuber_names = set()

with open(VTUBERS_LIST_FILE_PATH, 'r') as data_file:
    for line in data_file.readlines():
        if line.startswith("#"): # comment:
            continue
        vtuber_names.add(line.lower().strip())

vtuber_names = list(vtuber_names)

In [ ]:
# Check if all vtubers exists
counter = 0
for vtuber in vtuber_names:
    counter += 1
    print(f"[{counter}/{len(vtuber_names)}] Check vtuber {vtuber}")
    assert get_userdata_by_login(vtuber) is not None

In [5]:
print(f"Loaded {len(vtuber_names)} vtubers")

Loaded 253 vtubers


In [6]:
def get_cache_path():
    current_date = datetime.now()

    return os.path.join("data/", 
                        ".temp/",
                        f"followers_{current_date.year}_{current_date.month}_{current_date.day}.txt")


def load_followers_data(vtuber_names: List[str],
                        online: bool = False, 
                        num_workers: int = 32,
                        cache_path: str = get_cache_path()):
    if online:
        def load_followers_execution(vtuber_login: str) -> Tuple[str, Optional[List[FollowerData]]]:
            result = get_followers(vtuber_login, log=True, repeat_times=5, repeat_delay=1.0)
            if result is None:
                print("[ERROR]", "Failed loading vtuber", vtuber_login, "attempt", i)
            else:
                return vtuber_login, result


        vtuber_followers = {}
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = []

            for vtuber in vtuber_names:
                future = executor.submit(load_followers_execution, vtuber_login=vtuber)
                futures.append(future)
            
            for future in as_completed(futures):
                vtuber_login, followers_result = future.result()
                if followers_result is None:
                    print(f"Cannot load `{vtuber_login}` result")
                else:
                    empty_results = list(filter(lambda el: el.user is None, followers_result))
                    print(f"Got `{vtuber_login}` result, empty results: {len(empty_results)}/{len(followers_result)}")
                vtuber_followers[vtuber_login] = followers_result
        
        return vtuber_followers
    else:
        result = {}

        with open(cache_path, 'r') as cache_file:
            csvreader = csv.DictReader(cache_file, delimiter=',')

            for row in csvreader:
                vtuber = row['vtuber']
                followed_at = datetime.fromisoformat(row['followed_at'])
                if row['id']:
                    id              = int(row['id'])
                    login           = None if row['login'] == "None" else row['login']
                    created_at      = None if row['created_at'] == "None" else datetime.fromisoformat(row['created_at'])
                    deleted_at      = None if row['deleted_at'] == "None" else datetime.fromisoformat(row['deleted_at'])
                    follows_count   = int(row['follows_count'])
                    user_data = UserData(id=id,
                                        login=login,
                                        created_at=created_at,
                                        deleted_at=deleted_at,
                                        follows_count=follows_count)
                else:
                    user_data = None
                follower_data = FollowerData(user=user_data,
                                            followed_at=followed_at)
                
                if vtuber not in result.keys():
                    result[vtuber] = []
                result[vtuber].append(follower_data)
        
        return result


def cache_followers_data(vtuber_followers: Dict[str, List[FollowerData]],
                        cache_path: str = get_cache_path()):
    with open(cache_path, 'w') as cache_file:
        csvwriter = csv.writer(cache_file, delimiter=',')
        csvwriter.writerow(["vtuber",
                            "followed_at",
                            "id",
                            "login",
                            "created_at",
                            "deleted_at",
                            "follows_count"])

        for (vtuber, value) in vtuber_followers.items():
            for follower_data in value:
                if follower_data.user is None:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        "",
                                        "",
                                        "",
                                        "",
                                        ""])
                else:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        follower_data.user.id,
                                        follower_data.user.login,
                                        str(follower_data.user.created_at),
                                        str(follower_data.user.deleted_at),
                                        follower_data.user.follows_count])

In [7]:
vtuber_followers = load_followers_data(vtuber_names, online=False, num_workers=64, cache_path="./data/.temp/followers_2025_11_4.txt")

In [8]:
for vtuber in vtuber_names:
    assert vtuber_followers[vtuber] is not None

In [ ]:
cache_followers_data(vtuber_followers)

In [9]:
user_following = {}

for vtuber in vtuber_names:
    for follower in vtuber_followers[vtuber]:
        if follower.user is not None:
            user_name = follower.user.id
            if user_name not in user_following.keys():
                user_following[user_name] = []
            user_following[user_name].append(vtuber)

In [10]:
data = VtubersData(vtuber_names=vtuber_names, vtuber_followers=vtuber_followers, user_following=user_following)

## Графики роста аудитории

### График фолловов

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 

### График фолловеров без причин резких скачков

In [ ]:
IGNORED_VTUBERS = ['xkamysh', 'qchaan_9', 'rera_seal', "trixie_vox", "lerritay"]

sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 

### Рост уникальных фолловов

То есть не учитываются второй, третий и т.д. фолловы от одного и того же человека

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = {}
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.user is not None:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

auditory_data = auditory_data.values()
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

plt.hist(auditory_data, bins="auto", log=False, cumulative=False)
plt.show() 

### Уникальность витуберов

Посмотрим, какая часть аудитории витубера смотрит только его 

In [ ]:
print(len(data.user_following))

In [ ]:
unique_users = list(filter(lambda fd: len(fd[1]) <= 1, data.user_following.items()))
print(len(unique_users))

In [ ]:
vtuber_unique_followers_counter = {}
for (user, following) in unique_users:
    vtuber_name = following[0]
    vtuber_unique_followers_counter[vtuber_name] = vtuber_unique_followers_counter.get(vtuber_name, 0) + 1

vtuber_unique_followers_proportion = {}
for vtuber in data.vtuber_names:
    vtuber_unique_followers_proportion[vtuber] = 0
for (vtuber, unique_followers) in vtuber_unique_followers_counter.items():
    vtuber_unique_followers_proportion[vtuber] = unique_followers / len(data.vtuber_followers[vtuber]) * 100

sorted(vtuber_unique_followers_proportion.items(), 
       key=lambda el: el[1], 
       reverse=True)

In [ ]:
plt.hist(vtuber_unique_followers_proportion.values(), bins=100)
plt.show()

| Vtuber | Statistics | Description |
| --- | --- | --- |
| mad_demian | 3855/4519 85% | в хиатусе |
| bready_xo | 4625/5512 83% | в хиатусе |
| natakodo | 18171/22059 82% | в хиатусе |
| lastglance_ | 3371/4250 79% | в хиатусе |
| gishichi | 886/1264 70% | англо витубер??? |
| kashtan_mp4 | 2030/2982 68% | Ich weiß nicht |
| nyapuru | 8657/12733 67% | мужыыыыык |
| mukubae | 1613/2429 66% | в хиатусе |
| boopa | 8545/13370 63% | Ich weiß nicht |
| linamyth | 4732/7542 62% | в хиатусе |
| ananastya_nastya | 6511/10562 61% | Лига легендер |
| amikirisan | 5176/8408 61% | Ich weiß nicht |
| piwochan | 3942/6485 60% | Ich weiß nicht |
| naxajlka | 2142/3588 59% | Ich weiß nicht |
| lightfoxmanga | 4903/8276 59% | мужыыыыык |
| quinella_admin | 8864/15398 57% | Ich weiß nicht |
| vernirra | 10568/18474 57% | Ich weiß nicht |
| planyach | 26256/46423 56% | Ich weiß nicht |
| nekisekai | 7131/12849 55% | Ich weiß nicht |
| nnstreamercha | 2546/4724 53% | мужыыыыык |
| mamilokuchii | 8751/17174 50% | Лигалегендер, любимый у дедпи |
| yumekomoore | 15773/31344 50% | Ich weiß nicht |

In [ ]:
df = pd.DataFrame({
    "Followers": list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), data.vtuber_names)),
    "Unique followers (%)": list(map(lambda vtuber: vtuber_unique_followers_proportion[vtuber], data.vtuber_names)),
    "vtuber": data.vtuber_names
})
fig = px.scatter(df, x="Followers", y="Unique followers (%)", hover_name="vtuber")
fig.show()

### Кластеризация витуберов

#### Векторизация данных

In [ ]:
columns = data.user_following.keys()
user_id_to_column = {}
for (i, col) in zip(range(len(columns)), columns):
    user_id_to_column[col] = i

def vtuber2vec(vtuber: str) -> np.ndarray:
    result = np.zeros(len(data.user_following))
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            result[user_id_to_column[follower.user.id]] = 1
    
    return result

In [ ]:
dataset = []
for vtuber in data.vtuber_names:
    dataset.append(vtuber2vec(vtuber))
dataset = np.array(dataset)

#### Кластеризация и отрисвка

##### Косинусное расстояние

In [ ]:
labels = sklearn.cluster.HDBSCAN(min_cluster_size=2, max_cluster_size=100, metric="cosine").fit_predict(X=dataset)

In [ ]:
projection = sklearn.manifold.TSNE(metric="cosine", random_state=22).fit_transform(dataset)

In [ ]:
data_frame = pd.DataFrame({
    "x": projection[:, 0],
    "y": projection[:, 1],
    "Кластер": labels,
    "Имя": data.vtuber_names
})
data_frame["Кластер"] = data_frame["Кластер"].astype(str) #convert to string

In [ ]:
fig = px.scatter(data_frame, 
                 x="x", 
                 y="y",
                 color="Кластер", 
                 hover_name="Имя")
fig.show()

In [ ]:
def cosine_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return 1 - np.dot(vec1, vec2) / np.sqrt(np.sum(vec1**2)) / np.sqrt(np.sum(vec2**2))

def manhattan_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return np.sum(np.abs(vec1 - vec2))

def simple_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / (a + b - inter)

In [ ]:
MATRIX = np.zeros((len(data.vtuber_names), len(data.vtuber_names)))
METRIC = simple_metric

for i in range(len(data.vtuber_names)):
    print(f"{i})", data.vtuber_names[i])

    for j in range(len(data.vtuber_names)):
        vtuber1_vec = dataset[i]
        vtuber2_vec = dataset[j]
        distance = METRIC(vtuber1_vec, vtuber2_vec)
        MATRIX[i][j] = distance

In [ ]:
MAX_DISTANCE = 0.885
EDGES = []

for i in range(len(data.vtuber_names)):
    for j in range(len(data.vtuber_names)):
        if MATRIX[i][j] < MAX_DISTANCE and i < j:
            EDGES.append((i, j))

print("Edges:", len(EDGES))

fig = go.Figure()


for (i, j) in EDGES:
    projection1 = projection[i]
    projection2 = projection[j]
    fig.add_trace(go.Scatter(x=[projection1[0], projection2[0]],
                             y=[projection1[1], projection2[1]],
                             mode="lines",
                             marker_color="#222222"))
    
fig.add_trace(go.Scatter(x=projection[:, 0], 
                         y=projection[:, 1],
                         marker_color=labels, 
                         text=data.vtuber_names,
                         mode="markers"))
    
fig.show()

### Анализ количества фолловов у зрителей

#### Количество глобальных фолловов

In [ ]:
unique_users = []
used_logins = set()
for vtuber in data.vtuber_names:
    for follower_data in data.vtuber_followers[vtuber]:
        if follower_data.user is not None and follower_data.user.login not in used_logins:
            used_logins.add(follower_data.user.login)
            unique_users.append(follower_data.user)

In [ ]:
draw_data = unique_users
draw_data = list(map(lambda el: el.follows_count, draw_data))
draw_data = list(filter(lambda el: el < 100, draw_data))
df = pd.DataFrame({
    "Following": draw_data
})
fig = px.histogram(df, x="Following")
fig.show()

#### Количество локальных фолловов

In [ ]:
draw_data = data.user_following.items()
draw_data = list(map(lambda el: el[1], draw_data))
draw_data = list(map(lambda el: len(el), draw_data))
draw_data = list(filter(lambda el: el > 0, draw_data))
df = pd.DataFrame({
    "Following": draw_data
})
fig = px.histogram(df, x="Following")
fig.show()

#### Количество абсолютных уникалов среди фолловеров

In [ ]:
follows_count = []
abs_unique = []
for vtuber in data.vtuber_names:
    follows_count.append(len(data.vtuber_followers[vtuber]))
for vtuber in data.vtuber_names:
    vtuber_abs_unique = 0
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None and follower.user.follows_count == 1:
            vtuber_abs_unique += 1
    abs_unique.append(vtuber_abs_unique)
abs_unique = np.array(abs_unique)
follows_count = np.array(follows_count)

df = pd.DataFrame({
    "Absolutely unique followers (%)": abs_unique / follows_count * 100,
    "Followers": follows_count,
    "Vtuber": vtuber_names,
})
fig = px.scatter(df, x="Followers", y="Absolutely unique followers (%)", hover_name="Vtuber")
fig.show()

## Аналитика сообщений

In [11]:
@dataclass
class EmoteData:
    id: str


@dataclass
class MessageFragmentData:
    emote: Optional[EmoteData]
    text: Optional[str]
    mention: Optional[UserData]

    def as_text(self) -> str:
        if self.emote is not None:
            return f"<@{self.emote.id}>"
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return f"@{self.mention.login}"
        else:
            return ""

    def as_plain_text(self) -> str:
        if self.emote is not None:
            return f"<@{self.emote.id}>"
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return f"@{self.mention.login}"
        else:
            return ""


@dataclass
class MessageData:
    as_text: str
    as_plain_text: str
    fragments: List[MessageFragmentData]


@dataclass
class CommentData:
    id: str
    commenter: UserData
    contentOffsetSeconds: int
    message: MessageData


@dataclass
class VideoData:
    id: int
    title: str
    description: str
    created_at: datetime
    view_count: int
    comments: List[CommentData]


@dataclass
class VtuberVideosData:
    vtuber_names: List[str]
    vtuber_videos: Dict[str, List[VideoData]]

In [32]:
def json_to_fragment_data(json_data) -> MessageFragmentData:
    return MessageFragmentData(emote=None if json_data['emote'] is None else EmoteData(id=json_data['emote']['id']),
                               text=json_data['text'],
                               mention=None if json_data['mention'] is None else json_to_user_data(json_data['mention']))


def fragment_data_to_json(fragment_data: MessageFragmentData):
    return {
        'emote': None if fragment_data.emote is None else {
            'id': fragment_data.emote.id
        },
        'text': fragment_data.text,
        'mention': None if fragment_data.mention is None else user_data_to_json(fragment_data.mention)
    }


def json_to_message_data(json_data) -> MessageData:
    as_text = ""
    as_plain_text = ""
    fragments = []

    for fragment_json in json_data['fragments']:
        fragment = json_to_fragment_data(fragment_json)
        as_text += fragment.as_text()
        as_plain_text += fragment.as_plain_text()
        fragments.append(fragment)

    return MessageData(as_text=as_text,
                       as_plain_text=as_plain_text,
                       fragments=fragments)


def message_data_to_json(message_data: MessageData):
    return {
        'fragments': list(map(fragment_data_to_json, message_data.fragments))
    }


def json_to_comment_data(json_data) -> CommentData:
    return CommentData(id=json_data['id'],
                       commenter=None if json_data['commenter'] is None else json_to_user_data(json_data['commenter']),
                       contentOffsetSeconds=int(json_data['contentOffsetSeconds']),
                       message=json_to_message_data(json_data['message']))


def user_data_to_json(user_data: UserData):
    return {
        'id': user_data.id,
        'login': user_data.login,
        'createdAt': None if user_data.created_at is None else str(user_data.created_at),
        'deletedAt': None if user_data.deleted_at is None else str(user_data.deleted_at),
        'follows': {
            'totalCount': str(user_data.follows_count)
        }
    }


def comment_data_to_json(comment_data: CommentData):
    return {
        'id': comment_data.id,
        'commenter': None if comment_data.commenter is None else user_data_to_json(comment_data.commenter),
        'contentOffsetSeconds': str(comment_data.contentOffsetSeconds),
        'message': message_data_to_json(comment_data.message)
    }


def json_to_video_data(json_data) -> VideoData:
    comments = []
    for comment in json_data['comments']['edges']:
        comments.append(json_to_comment_data(comment['node']))

    return VideoData(id=int(json_data['id']),
                     title=json_data['title'],
                     description=json_data['description'],
                     created_at=None if json_data['createdAt'] is None else datetime.fromisoformat(json_data['createdAt']),
                     view_count=int(json_data['viewCount']),
                     comments=comments)


def video_data_to_json(video_data: VideoData):
    return {
        'id': str(video_data.id),
        'title': video_data.title,
        'description': video_data.description,
        'createdAt': None if video_data.created_at is None else str(video_data.created_at),
        'viewCount': str(video_data.view_count),
        'comments': {
            'edges': list(map(
                lambda comment_data: {
                    'node': comment_data_to_json(comment_data)
                }, 
                video_data.comments))
        },
    }


def join_video_data(left: Optional[VideoData], right: VideoData) -> VideoData:
    if left is None:
        return right
    else:
        assert left.id == right.id

        comments = []
        comments.extend(left.comments)
        comments.extend(right.comments)

        return VideoData(id=left.id,
                         title=left.title,
                         description=left.description,
                         created_at=left.created_at,
                         view_count=left.view_count,
                         comments=comments)

In [33]:
import requests
import time
import json
import pprint


API_URL = "https://gql.twitch.tv/gql"
API_CLIENT_ID = "kd1unb4b3q4t58fwlpcbzcbnm76a8fp"
API_REQUEST_DATA = """
query fetchVideoData($id: ID, $first: Int, $after: Cursor) {
    video(id: $id) {
        id
        title
        description
        createdAt
        viewCount
        comments(first: $first, after: $after) {
            edges {
                cursor
                node {
                    id
                    commenter {
                        id
                        login
                        createdAt
                        deletedAt
                        follows {
                            totalCount
                        }
                    }
                    contentOffsetSeconds
                    message {
                        fragments {
                            emote {
                                id
                            }
                            mention {
                                id
                                login
                                createdAt
                                deletedAt
                                follows {
                                    totalCount
                                }
                            }
                            text
                        }
                    }
                }
            }
            pageInfo {
                hasNextPage
            }
        }
    }
}
"""


def send_request(session: requests.Session, 
                 id: int, 
                 cursor: str,
                 log: bool,
                 repeat_times: int,
                 repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA,
                'variables': {
                    'id': str(id),
                    'first': 100,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_video_data(id: int, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0) -> Optional[VideoData]:
    cache_path = f"./data/.temp/video/{id}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            return json_to_video_data(json.load(cache_file))

    session = requests.Session()
    cursor = None
    result = None

    while True:
        response = send_request(session, id, cursor, log, repeat_times, repeat_delay)

        if response is None:
            if log:
                print("[ERROR]", "Failed load", id)
            return None

        response_data = json.loads(response.text)
        result = join_video_data(result, json_to_video_data(response_data['data']['video']))
        

        if log:
            if len(result.comments) > 0:
                print(f"[LOG|{id}]", "Loaded", 
                      result.comments[-1].contentOffsetSeconds // 60 // 60, "hours", 
                      result.comments[-1].contentOffsetSeconds // 60 % 60, "minutes", 
                      result.comments[-1].contentOffsetSeconds % 60, "seconds")
            else:
                print(f"[LOG|{id}] Loaded ...")

        if len(response_data['data']['video']['comments']['edges']) == 0:
            break
        else:
            cursor = response_data['data']['video']['comments']['edges'][-1]['cursor']

        if not response_data['data']['video']['comments']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print('[LOG]', "caching")
    with open(cache_path, 'w') as cache_file:
        json.dump(video_data_to_json(result), cache_file, indent=4)

    return result
        


In [43]:
API_REQUEST_DATA_2 = """
query fetchVideoData($login: String, $first: Int, $after: Cursor) {
    user(login: $login) {
        videos(first: $first, after: $after, type: ARCHIVE, sort: TIME) {
            totalCount
            pageInfo {
                hasNextPage
            }
            edges {
                cursor
                node {
                    id
                }
            }
        }
    }
}
"""


def send_request_2(session: requests.Session, 
                   login: str, 
                   cursor: str,
                   log: bool,
                   repeat_times: int,
                   repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA_2,
                'variables': {
                    'login': login,
                    'first': 20,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_streamer_videos(login: str, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0, cacheonly: bool = False) -> Optional[List[int]]:
    result = []
    cache_path = f"./data/.temp/videos/{login}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            result = json.load(cache_file)

    if cacheonly:
        return result

    session = requests.Session()
    cursor = None

    while True:
        response = send_request_2(session, login, cursor, log, repeat_times, repeat_delay)
        assert response is not None

        response_data = json.loads(response.text)

        for video_data in response_data['data']['user']['videos']['edges']:
            cursor = video_data['cursor']
            id = int(video_data['node']['id'])

            if id not in result:
                result.append(id)
        
        if not response_data['data']['user']['videos']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print("[LOG]", "Caching result")
    with open(cache_path, 'w') as cache_file:
        json.dump(result, cache_file, indent=4)

    return result

In [39]:
videos_cache = {}

In [42]:
def load_vtuber_video(vtuber_login: str, video_id: int) -> Tuple[str, Optional[VideoData]]:
    assert load_video_data(video_id, log=True, repeat_times=10, repeat_delay=5.0) is not None


vtuber_videos = {}
with ThreadPoolExecutor(max_workers=64) as executor:
    futures = []

    for vtuber in data.vtuber_names:
        streamer_videos = load_streamer_videos(vtuber, log=True)
        assert streamer_videos is not None
        for video_id in streamer_videos:
            future = executor.submit(load_vtuber_video, vtuber_login=vtuber, video_id=video_id)
            futures.append(future)
    
    for future in as_completed(futures):
        future.result()

[LOG] Loading from the cache
[LOG] Caching result
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG][LOG] Loading from the cache
 Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Caching result
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[

In [45]:
# @dataclass
# class VtuberVideosData:
#     vtuber_names: List[str]
#     vtuber_videos: Dict[str, List[VideoData]]

vtuber_videos = {}
for vtuber in data.vtuber_names:
    vtuber_videos[vtuber] = []

    for video_id in load_streamer_videos(vtuber, log=True, cacheonly=True):
        video_data = load_video_data(video_id)
        vtuber_videos[vtuber].append(video_data)

[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading from the cache
[LOG] Loading 

In [46]:
vvdata = VtuberVideosData(vtuber_names=data.vtuber_names,
                          vtuber_videos=vtuber_videos)

In [ ]:
vvdata